# Bimodal mixture on nPRF R²: separating signal from noise voxels

Instead of an arbitrary `R² > 0.02` cutoff, fit a **2-component mixture** to the per-voxel
nPRF R² distribution:

- **noise** component — voxels whose R² is pure finite-sample overfitting
- **signal** component — voxels carrying real numerosity-tuned signal

From the fit we read off exactly what we want per subject:

| quantity | where it comes from |
|---|---|
| mean R² of the *signal* voxels | `fit['signal_mean_r2']` |
| how many voxels carry signal | `fit['signal_weight'] * n`, or a posterior / FDR count |
| a principled R² threshold | `r2_fdr_threshold(fit, alpha)` — tail-FDR ≤ α |
| per-voxel confidence | `posterior_p_signal(r2, fit)` = P(signal \| R²) |

## Implementation

This uses Gilles' implementation in **braincoder**, on branch
[`feature/f-beta-r2-mixture`](https://github.com/Gilles86/braincoder/tree/feature/f-beta-r2-mixture)
(`braincoder/utils/stats.py`), documented in its
[Lesson 8 tutorial](https://github.com/Gilles86/braincoder/blob/feature/f-beta-r2-mixture/docs/tutorial/lesson8_r2_fdr.rst).
**This notebook only works if your braincoder clone is on that branch** — on any
machine you run it (laptop *and* server):

```bash
cd <your braincoder clone> && git fetch origin
git checkout -b f-beta-r2-mixture origin/feature/f-beta-r2-mixture
```

The setup cell raises a clear `ImportError` with these instructions if it isn't.

Two mixture flavours are available; this notebook runs **both** and compares them:

1. **`fit_r2_mixture`** (default) — 2-component Gaussian mixture on `logit(R²)`.
   The logit is what makes it work: on raw R² both components sit on a bounded
   [0,1] support with the noise mode pinned at the boundary, which makes naive
   Beta-mixture EM unstable. On the logit scale the noise spike is a clean
   Gaussian and the signal tail stays Gaussian-shaped.
2. **`fit_r2_f_beta_mixture`** — noise is `Beta(d1/2, d2/2)`, i.e. F-distributed
   under the classical null, with `d1` **fixed to the number of free per-voxel PRF
   parameters**. This anchors the null's tail shape to model complexity and is the
   documented fallback when flavour 1 goes degenerate.

> Gilles' tutorial flags this pathology explicitly: if `r2_fdr_threshold` returns
> `inf`, or the signal PDF comes out nearly flat, the logit-Gaussian mixture has
> collapsed — fit per-ROI rather than whole-brain, or use the F+Beta variant.
> The summary table below reports this per voxel set so you can see it happening.

## Setup

`braincoder.utils.__init__` imports tensorflow, which is broken in the local
`numrefields` env. The mixture code needs only numpy/scipy/sklearn, so we load
`stats.py` directly by path and skip the package `__init__` entirely. On a machine
where tensorflow works, the plain `from braincoder.utils.stats import ...` is used.

In [ ]:
import os
import os.path as op
import importlib.util

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns

try:
    # normal route (works wherever tensorflow imports cleanly, e.g. the server)
    from braincoder.utils import stats as bcstats
except Exception as e:
    # tensorflow-free route: load stats.py directly, bypassing braincoder/utils/__init__.py
    # (its math.py imports tensorflow, which the mixture code does not need).
    # Locate the package wherever it happens to be installed -- do NOT hardcode a path,
    # this notebook runs on both the laptop and the server.
    print(f'plain import failed ({type(e).__name__}), loading stats.py directly')
    _bc = importlib.util.find_spec('braincoder')
    if _bc is None or not _bc.submodule_search_locations:
        raise ImportError('braincoder not importable at all -- is it installed?') from e
    _stats_fn = op.join(list(_bc.submodule_search_locations)[0], 'utils', 'stats.py')
    if not op.exists(_stats_fn):
        raise ImportError(f'{_stats_fn} not found') from e
    _spec = importlib.util.spec_from_file_location('bcstats', _stats_fn)
    bcstats = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(bcstats)

# fail loudly and early if braincoder is on a branch without the mixture code,
# rather than 200 lines later with a confusing AttributeError
_missing = [f for f in ('fit_r2_mixture', 'fit_r2_f_beta_mixture', 'r2_fdr_threshold')
            if not hasattr(bcstats, f)]
if _missing:
    raise ImportError(
        f'braincoder is missing {_missing}. The R2-mixture code lives on branch '
        "'feature/f-beta-r2-mixture'; check out that branch in your braincoder clone:\n"
        '    cd <braincoder clone> && git fetch origin && \\\n'
        '        git checkout -b f-beta-r2-mixture origin/feature/f-beta-r2-mixture')

fit_r2_mixture         = bcstats.fit_r2_mixture
posterior_p_signal     = bcstats.posterior_p_signal
r2_fdr_threshold       = bcstats.r2_fdr_threshold
plot_r2_mixture        = bcstats.plot_r2_mixture
fit_r2_f_beta_mixture  = bcstats.fit_r2_f_beta_mixture
r2_fdr_threshold_f_beta = bcstats.r2_fdr_threshold_f_beta
plot_r2_f_beta_mixture = bcstats.plot_r2_f_beta_mixture

sns.set_style('ticks')
print('braincoder mixture functions loaded')

## Configuration

`SPACE` switches the whole notebook between **`fsaverage5`** surface vertices
(same maps as `parietal_patterns/prep_results/paperDD_neuralFig1.ipynb`) and
**`T1w`** native volume voxels.

Note on data location: the per-subject `encoding_model.denoise` maps live on the
server (`/mnt_03/ds-dnumrisk`). The local copy at `~/data/ds-dnumrisk` has only
`encoding_model.denoise/averages/` — no per-subject files — so **this notebook
needs the server path for the stim1 nPRF maps**. `LOCAL_FALLBACK` below points at
the one dataset that *is* complete locally (`encoding_model_stim2.denoise`, T1w,
sub-01) so the notebook can at least be smoke-tested offline.

In [ ]:
# ---- what to analyse -------------------------------------------------------
SPACE      = 'fsaverage5'          # 'fsaverage5' (surface) | 'T1w' (native volume)
NPRF_KEY   = 'encoding_model.denoise'   # stim1 nPRF fits, as in paperDD_neuralFig1
SESSION    = 1
SUBJECT    = 1

ALPHA      = 0.05   # tail-FDR level for the R² threshold
P_SIGNAL_CUT = 0.95 # posterior cutoff for "this voxel carries signal"

# free per-voxel parameters of the encoding model -> the F-noise numerator dof.
# braincoder.models.GaussianPRF.parameter_labels == ['mu', 'sd', 'amplitude', 'baseline']
N_PRF_PARAMS = 4

# ---- where the data is -----------------------------------------------------
BIDS_REMOTE = '/mnt_03/ds-dnumrisk'
BIDS_LOCAL  = '/Users/mrenke/data/ds-dnumrisk'

bids_folder = BIDS_REMOTE if op.exists(BIDS_REMOTE) else BIDS_LOCAL
if bids_folder == BIDS_LOCAL:
    print(f'!! server not mounted, falling back to {BIDS_LOCAL}')
    print('   -> per-subject encoding_model.denoise maps are NOT there; '
          'set the LOCAL_FALLBACK cell below to smoke-test.')
print('bids_folder:', bids_folder)

# subjects, matching paperDD_neuralFig1.ipynb
subList = np.arange(1, 67)
subList = subList[subList != 3]

group_fn = op.join(bids_folder, 'group_assignment.csv')
group_df = pd.read_csv(group_fn).set_index('subject').loc[subList] if op.exists(group_fn) else None
print('n subjects:', len(subList))

## Loaders

`load_r2` returns the R² map over the *full* space (20484 fsaverage5 vertices, or
the flattened volume), and `load_masks` returns boolean masks defined on that same
index — so every voxel set is just `r2[mask]`.

- **fsaverage5** — R² from the two `*_space-fsaverage5_hemi-{L,R}.func.gii` files,
  concatenated L→R. Cortex mask from the Destrieux atlas (drops medial wall /
  unknown, same as `parietal_patterns.utils.surfaces.get_basic_mask`); NPC from
  `derivatives/surface_masks/desc-NPC_{L,R}_space-fsaverage5_hemi-{lh,rh}.label.gii`.
- **T1w** — R² from `*_desc-r2.optim_space-T1w_pars.nii.gz`; brain mask from
  fmriprep; NPC from `derivatives/ips_masks/`, resampled (nearest) onto the
  functional grid — those masks are stored at anatomical resolution and do *not*
  share the functional affine.

In [ ]:
def load_r2(subject, space, bids_folder, key=NPRF_KEY, session=SESSION):
    """Per-voxel/vertex nPRF R². Returns (r2_flat, ref_img). ref_img is None on surfaces."""
    sub = f'sub-{subject:02d}'
    func = op.join(bids_folder, 'derivatives', key, sub, f'ses-{session}', 'func')

    if space == 'fsaverage5':
        hemis = [nib.load(op.join(
            func, f'{sub}_ses-{session}_desc-r2.optim.nilearn_'
                  f'space-fsaverage5_hemi-{h}.func.gii')).agg_data() for h in ['L', 'R']]
        return np.concatenate(hemis).astype(float), None

    elif space == 'T1w':
        img = nib.load(op.join(
            func, f'{sub}_ses-{session}_desc-r2.optim_space-T1w_pars.nii.gz'))
        return img.get_fdata().ravel().astype(float), img

    raise ValueError(f"space must be 'fsaverage5' or 'T1w', got {space!r}")


def load_masks(subject, space, bids_folder, ref_img=None, session=SESSION):
    """Boolean masks on the same flat index as load_r2. Keys: 'whole brain', 'NPC', 'NPC_L', 'NPC_R'."""
    sub = f'sub-{subject:02d}'

    if space == 'fsaverage5':
        from nilearn import datasets
        atlas = datasets.fetch_atlas_surf_destrieux()
        labeling = np.concatenate([atlas['map_left'], atlas['map_right']])
        drop = [atlas['labels'].index(r) for r in (b'Medial_wall', b'Unknown')]
        cortex = ~np.isin(labeling, drop)

        mdir = op.join(bids_folder, 'derivatives', 'surface_masks')
        npc = {}
        for side, h in [('L', 'lh'), ('R', 'rh')]:
            g = nib.load(op.join(
                mdir, f'desc-NPC_{side}_space-fsaverage5_hemi-{h}.label.gii')).agg_data()
            npc[side] = np.bool_(g)
        zL, zR = np.zeros_like(npc['L']), np.zeros_like(npc['R'])
        npc_L = np.concatenate([npc['L'], zR])
        npc_R = np.concatenate([zL, npc['R']])

    elif space == 'T1w':
        from nilearn.image import resample_to_img, index_img
        bm = nib.load(op.join(
            bids_folder, 'derivatives', 'fmriprep', sub, f'ses-{session}', 'func',
            f'{sub}_ses-{session}_task-magjudge_run-1_space-T1w_desc-brain_mask.nii.gz'))
        cortex = bm.get_fdata().ravel() > 0   # whole brain, not cortex, in this space

        mdir = op.join(bids_folder, 'derivatives', 'ips_masks', sub)
        out = {}
        for side in ['L', 'R']:
            m = nib.load(op.join(mdir, f'{sub}_space-T1w_desc-NPC_{side}.nii.gz'))
            if m.ndim == 4:                       # stored as (x, y, z, 1)
                m = index_img(m, 0)
            # these masks are int64 on disk, which nibabel refuses to write back out
            # during resampling -> recast to float32 first
            m = nib.Nifti1Image(m.get_fdata().astype(np.float32), m.affine)
            # anatomical-resolution masks: resample onto the functional grid
            m = resample_to_img(m, ref_img, interpolation='nearest')
            out[side] = m.get_fdata().ravel() > 0.5
        npc_L, npc_R = out['L'], out['R']

    else:
        raise ValueError(f"space must be 'fsaverage5' or 'T1w', got {space!r}")

    return {'whole brain': cortex,
            'NPC':         (npc_L | npc_R) & cortex,
            'NPC_L':       npc_L & cortex,
            'NPC_R':       npc_R & cortex}

In [ ]:
r2_full, ref_img = load_r2(SUBJECT, SPACE, bids_folder)
masks = load_masks(SUBJECT, SPACE, bids_folder, ref_img=ref_img)

voxel_sets = {name: r2_full[m] for name, m in masks.items()}

print(f'sub-{SUBJECT:02d}  space={SPACE}  R² map: {r2_full.shape[0]} elements\n')
for name, vals in voxel_sets.items():
    ok = np.isfinite(vals) & (vals > 0) & (vals < 0.99)
    print(f'{name:<12} n={vals.size:>7}  usable={ok.sum():>7}  '
          f'median R²={np.nanmedian(vals):.4f}  max={np.nanmax(vals):.3f}')

### Local smoke-test fallback (optional)

Run this **only** if the server isn't mounted. It swaps in the one complete local
dataset — `encoding_model_stim2.denoise`, sub-01, T1w — so the machinery below
executes. Note this is *stim2 within-sample* R², a different and much weaker map
than the stim1 fits, so don't read the numbers as results.

In [ ]:
# if bids_folder == BIDS_LOCAL:
#     SPACE = 'T1w'
#     r2_full, ref_img = load_r2(1, 'T1w', BIDS_LOCAL, key='encoding_model_stim2.denoise')
#     masks = load_masks(1, 'T1w', BIDS_LOCAL, ref_img=ref_img)
#     voxel_sets = {name: r2_full[m] for name, m in masks.items()}
#     print({k: int(v.size) for k, v in voxel_sets.items()})

## Fit the mixture

`fit_one` runs both flavours on one voxel set and pulls out the numbers of
interest. Three different counts of "voxels with signal" are reported, because
they answer slightly different questions:

- `n_signal_weight` = `w_signal · n` — the mixture's own estimate of how many
  voxels belong to the signal population. **This is the least biased count** and
  the one to quote as "number of voxels with signal".
- `n_signal_posterior` = `#(P(signal|R²) ≥ 0.95)` — voxels we are individually
  confident about. Conservative; use it when you need to *select* voxels.
- `n_signal_fdr` = `#(R² ≥ threshold)` — voxels surviving the tail-FDR cut.

They diverge when the components overlap: the weight-based count includes
low-R² signal voxels that no per-voxel rule can confidently claim.

In [ ]:
def fit_one(vals, label, alpha=ALPHA, d1_noise=N_PRF_PARAMS, p_cut=P_SIGNAL_CUT):
    """Fit both mixture flavours to one voxel set; return (summary_row, fits)."""
    vals = np.asarray(vals, dtype=float).ravel()
    usable = np.isfinite(vals) & (vals > 0) & (vals < 0.99)
    row = {'voxel_set': label, 'n_voxels': int(vals.size), 'n_usable': int(usable.sum())}
    fits = {}

    # --- flavour 1: logit-Gaussian ---
    try:
        fit = fit_r2_mixture(vals)
        thr = r2_fdr_threshold(fit, alpha=alpha)
        p_sig = posterior_p_signal(vals, fit)
        fits['gmm_logit'] = fit
        row.update({
            'noise_mean_r2':  fit['noise_mean_r2'],
            'noise_weight':   fit['noise_weight'],
            'signal_mean_r2': fit['signal_mean_r2'],
            'signal_weight':  fit['signal_weight'],
            'n_signal_weight':    int(round(fit['signal_weight'] * fit['n_voxels'])),
            'n_signal_posterior': int(np.nansum(p_sig >= p_cut)),
            'r2_threshold':       thr,
            'n_signal_fdr':       int(np.sum(vals[usable] >= thr)) if np.isfinite(thr) else 0,
            # mean R² among the voxels we actually call signal
            'mean_r2_of_signal_voxels': float(np.nanmean(vals[np.nan_to_num(p_sig) >= p_cut]))
                                        if np.nansum(p_sig >= p_cut) else np.nan,
            'degenerate': not np.isfinite(thr),
        })
    except ValueError as e:
        row.update({'gmm_error': str(e), 'degenerate': True})

    # --- flavour 2: F(d1 fixed) + Beta, on the raw R² scale ---
    try:
        fb = fit_r2_f_beta_mixture(vals, d1_noise=d1_noise)
        thr_fb = r2_fdr_threshold_f_beta(fb, alpha=alpha)
        fits['f_beta'] = fb
        row.update({
            'fb_noise_mean_r2':  fb['noise_mean_r2'],
            'fb_signal_mean_r2': fb['signal_mean_r2'],
            'fb_signal_weight':  fb['signal_weight'],
            'fb_n_signal_weight': int(round(fb['signal_weight'] * fb['n_voxels'])),
            'fb_r2_threshold':   thr_fb,
            'fb_n_signal_fdr':   int(np.sum(vals[usable] >= thr_fb)) if np.isfinite(thr_fb) else 0,
        })
    except ValueError as e:
        row['fb_error'] = str(e)

    return row, fits


rows, all_fits = [], {}
for name, vals in voxel_sets.items():
    row, fits = fit_one(vals, name)
    rows.append(row)
    all_fits[name] = fits

summary = pd.DataFrame(rows).set_index('voxel_set')
summary

In [ ]:
# headline numbers, per voxel set
cols = ['n_usable', 'noise_mean_r2', 'noise_weight',
        'signal_mean_r2', 'signal_weight',
        'n_signal_weight', 'n_signal_posterior', 'n_signal_fdr', 'r2_threshold']
display(summary[[c for c in cols if c in summary]].round(4))

for name in summary.index:
    if summary.loc[name].get('degenerate', False):
        print(f'!! {name}: logit-Gaussian mixture is degenerate (threshold = inf). '
              f'Use the F+Beta columns, or restrict to an ROI.')

## Diagnostic plots

The grey histogram is the empirical R² distribution (logit-scaled x-axis for the
Gaussian flavour, raw R² for F+Beta), blue/red are the fitted noise and signal
component PDFs, the dotted line is the α-level FDR threshold.

Read these before trusting the numbers. A signal PDF that lies nearly flat, or
one that sits on top of the noise component, means the split is not real.

In [ ]:
names = [n for n in voxel_sets if 'gmm_logit' in all_fits.get(n, {})]

fig, axes = plt.subplots(1, len(names), figsize=(5.2 * len(names), 3.8), squeeze=False)
for ax, name in zip(axes[0], names):
    plot_r2_mixture(all_fits[name]['gmm_logit'], r2=voxel_sets[name],
                    alpha=ALPHA, ax=ax, title=f'{name}  (logit-Gaussian)')
    ax.set_yscale('log')
fig.suptitle(f'sub-{SUBJECT:02d} — {SPACE} — {NPRF_KEY}', y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
names_fb = [n for n in voxel_sets if 'f_beta' in all_fits.get(n, {})]

fig, axes = plt.subplots(1, len(names_fb), figsize=(5.2 * len(names_fb), 3.8), squeeze=False)
for ax, name in zip(axes[0], names_fb):
    plot_r2_f_beta_mixture(all_fits[name]['f_beta'], r2=voxel_sets[name],
                           alpha=ALPHA, ax=ax, title=f'{name}  (F+Beta, d1={N_PRF_PARAMS})')
fig.suptitle(f'sub-{SUBJECT:02d} — {SPACE} — model-anchored noise', y=1.03)
fig.tight_layout()
plt.show()

## Per-voxel posterior map

`posterior_p_signal` grades every voxel by P(signal | R²) instead of hard-thresholding.
Written back out in the input space so it can be viewed on the surface / in the volume.

In [ ]:
SET_FOR_MAP = 'whole brain'
fit = all_fits[SET_FOR_MAP]['gmm_logit']

# posterior over the full map, evaluated with the fit from SET_FOR_MAP
p_signal_full = posterior_p_signal(r2_full, fit)
thr = r2_fdr_threshold(fit, alpha=ALPHA)

print(f'fit on: {SET_FOR_MAP}')
print(f"signal component: mean R² = {fit['signal_mean_r2']:.4f}, "
      f"weight = {fit['signal_weight']:.3f}")
print(f'α={ALPHA} FDR threshold: R² ≥ {thr:.4f}' if np.isfinite(thr)
      else f'α={ALPHA} FDR threshold: inf (degenerate)')

for name, m in masks.items():
    n = int(np.nansum(p_signal_full[m] >= P_SIGNAL_CUT))
    r2_sig = r2_full[m][np.nan_to_num(p_signal_full[m]) >= P_SIGNAL_CUT]
    print(f'  {name:<12} {n:>6} voxels with P(signal) ≥ {P_SIGNAL_CUT}'
          + (f'   mean R² = {r2_sig.mean():.4f}' if n else ''))

fig, ax = plt.subplots(figsize=(5, 3.5))
ok = np.isfinite(r2_full) & np.isfinite(p_signal_full)
ax.plot(r2_full[ok], p_signal_full[ok], '.', ms=1, alpha=.3, color='0.4')
if np.isfinite(thr):
    ax.axvline(thr, color='k', ls=':', label=f'FDR α={ALPHA}')
ax.axhline(P_SIGNAL_CUT, color='r', ls='--', lw=1, label=f'P ≥ {P_SIGNAL_CUT}')
ax.set(xlabel='R²', ylabel='P(signal | R²)', xscale='log')
ax.legend(fontsize=8)
sns.despine(ax=ax)
plt.show()

In [ ]:
SAVE_MAP = False   # flip to True to write the posterior map to derivatives/

if SAVE_MAP:
    out_dir = op.join(bids_folder, 'derivatives', f'{NPRF_KEY}.r2mixture',
                      f'sub-{SUBJECT:02d}', f'ses-{SESSION}', 'func')
    os.makedirs(out_dir, exist_ok=True)
    stem = f'sub-{SUBJECT:02d}_ses-{SESSION}_desc-psignal'

    if SPACE == 'fsaverage5':
        for hemi, arr in zip(['L', 'R'], np.split(p_signal_full, 2)):
            g = nib.gifti.GiftiImage(darrays=[
                nib.gifti.GiftiDataArray(arr.astype(np.float32))])
            fn = op.join(out_dir, f'{stem}_space-fsaverage5_hemi-{hemi}.func.gii')
            nib.save(g, fn)
            print('wrote', fn)
    else:
        img = nib.Nifti1Image(
            p_signal_full.reshape(ref_img.shape).astype(np.float32),
            ref_img.affine, ref_img.header)
        fn = op.join(out_dir, f'{stem}_space-T1w.nii.gz')
        img.to_filename(fn)
        print('wrote', fn)

## Group level

Loop the same fit over all subjects and collect one row per subject × voxel set.
Slow-ish on the surface (the Destrieux atlas fetch is cached after the first call);
the mixture fit itself is a few hundred ms per voxel set.

In [ ]:
RUN_GROUP = False   # flip to True

if RUN_GROUP:
    group_rows = []
    for sub in subList:
        try:
            r2_s, ref_s = load_r2(sub, SPACE, bids_folder)
            masks_s = load_masks(sub, SPACE, bids_folder, ref_img=ref_s)
        except FileNotFoundError as e:
            print(f'sub-{sub:02d}: missing ({op.basename(str(e).split()[-1])})')
            continue
        for name, m in masks_s.items():
            row, _ = fit_one(r2_s[m], name)
            row['subject'] = sub
            group_rows.append(row)
        print(f'sub-{sub:02d} done', end='\r')

    df_group = (pd.DataFrame(group_rows)
                  .set_index(['subject', 'voxel_set'])
                  .sort_index())
    if group_df is not None:
        df_group = df_group.join(group_df, on='subject')
    display(df_group.head(12))

In [ ]:
if RUN_GROUP and group_df is not None:
    # does the signal population differ between Control and Dyscalculia?
    measures = ['signal_mean_r2', 'signal_weight', 'n_signal_weight', 'n_signal_posterior']
    d = df_group.xs('NPC', level='voxel_set').reset_index()
    d['group_label'] = d['group'].map({0: 'Control', 1: 'Dyscalculia'})

    fig, axes = plt.subplots(1, len(measures), figsize=(3.2 * len(measures), 3),
                             constrained_layout=True)
    for ax, msr in zip(axes, measures):
        sns.boxplot(data=d, x='group_label', y=msr, ax=ax, width=.5,
                    showfliers=False, palette='Set2')
        sns.stripplot(data=d, x='group_label', y=msr, ax=ax, color='.25', size=3, alpha=.6)
        ax.set(xlabel='', title=msr)
    sns.despine(fig=fig, offset=4, trim=True)
    plt.show()

In [ ]:
SAVE_GROUP = False

if RUN_GROUP and SAVE_GROUP:
    out_fn = op.join(bids_folder, 'derivatives', 'phenotype',
                     f'nPRF_r2mixture_space-{SPACE}.csv')
    os.makedirs(op.dirname(out_fn), exist_ok=True)
    df_group.to_csv(out_fn)
    print('wrote', out_fn)